# Hotel cancellation prediction

Explore hotel bookings and prepare data to predict cancellations.

Import the data preparation code from the `src` folder.

In [1]:
import sys
from pathlib import Path
from importlib import import_module

# Import the local source when running from the project or notebook directory.
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "hotel_cancellation").is_dir()
)
sys.path.insert(0, str(project_root / "src"))

print(f"Python: {sys.executable}")
preprocessing = import_module("hotel_cancellation.0_preprocessing")


Python: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12


## 1. Data

Load the CSV. Check the number of rows and columns and view the first five bookings.

In [2]:
df = preprocessing.load_data()
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NULL,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03


Check column types, the five columns with the most missing values, and the share of bookings: `0` = not canceled, `1` = canceled.

In [3]:
df.info()
display(df.isna().sum().sort_values(ascending=False).head())
display(df["is_canceled"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

country               488
children                4
hotel                   0
agent                   0
reserved_room_type      0
dtype: int64

is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64

Prepare `X` (information for the model) and `y` (whether the booking was canceled). Remove status columns that reveal the answer, clean text, and add total nights and guests.

In [4]:
X, y = preprocessing.prepare_features(df)
print(X.shape, y.shape)

(119390, 31) (119390,)


In [5]:
# Inspect the feature matrix
print("Number of observations:", X.shape[0])
print("Number of features:", X.shape[1])

# Separate numerical and categorical columns
numerical_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Number of observations: 119390
Number of features: 31

Numerical features: 19
Categorical features: 12


**Dimension of Feature Space** : $$ X \in \mathbb{R}^{11390 \times 31} $$

## 2. Training


### 2.1 Logistic Regression

We wish to establish a baseline classification model via Logistic Regression; however, we must first partition the dataset into training (80%) and test (20%) subsets. 

(Note the use of *stratification* parameter in the "train_test_split" function to preserve the proportion of canceled and non-canceled reservations)

In [6]:
# Import helper function
from sklearn.model_selection import train_test_split

# Split data 
X_train, X_test, y_train, y_test, = train_test_split(
    X, y,
    test_size=0.2, # implies train_size=0.8
    random_state=72, # seed set to ensure reproduceability 
    stratify=y
)


print("Training observations: ", X_train.shape[0])
print("Testing observations: ", X_test.shape[0])


Training observations:  95512
Testing observations:  23878


### Mathematical Model (Sigmoid)

Estimating the probability of cancellation using the sigmoid function:
 
$$ 
P(Y=1\mid X=x)=\frac{1}{1+e^{-(\beta_0+\boldsymbol{\beta}^{T}x)}}
$$


In [7]:
# Import helper function(s) / model
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(
    preprocessing.build_preprocessor(),
    LogisticRegression(max_iter=1000, random_state=72)
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression Training Successfully Complete!")

Logistic Regression Training Successfully Complete!


## 3. Evaluation
